# Mounting Google Drive

In [3]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import math
from scipy import stats
import numpy as np
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=DeprecationWarning)


In [5]:
cd /content/drive/MyDrive/data

/content/drive/MyDrive/data


In [6]:
ls

airlines.csv  airports.csv  flights.csv


# Problem 1

In [7]:
flights = Table.read_table('flights.csv')
airlines = Table.read_table('airlines.csv')
airports = Table.read_table('airports.csv')

/usr/local/lib/python3.11/dist-packages/datascience/tables.py:163: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pandas.read_csv(filepath_or_buffer, *args, **vargs)


In [8]:
unique_airlines = flights.column('AIRLINE')
counts = {airline: np.count_nonzero(unique_airlines == airline) for airline in set(unique_airlines)}
count_by_airline = Table().with_columns('AIRLINE', list(counts.keys()), 'count', list(counts.values()))
count_by_airline = count_by_airline.join('AIRLINE', airlines, 'IATA_CODE')
count_by_airline = count_by_airline.sort('count', descending=True)

In [9]:
AmFlights = (flights.where('AIRLINE', are.contained_in('AA MQ')))

In [10]:
airport_counts = AmFlights.group('ORIGIN_AIRPORT').sort('count', descending=True)
airport_counts

ORIGIN_AIRPORT,count
DFW,187873
ORD,113871
MIA,53625
CLT,41867
LAX,32922
LGA,28274
PHX,28214
PHL,21102
JFK,19212
DCA,18621


In [13]:
average_airtime = AmFlights.select('AIR_TIME', 'ORIGIN_AIRPORT').group('ORIGIN_AIRPORT', np.nanmean)
average_airtime = average_airtime.join('ORIGIN_AIRPORT', airport_counts, 'ORIGIN_AIRPORT')
average_airtime = average_airtime.join('ORIGIN_AIRPORT', airports, 'IATA_CODE')

In [17]:
airtime_bins = [0, 60, 120, 180, 240, 300]  # Bins for airtime in minutes
airtime_colors = np.array(['blue', 'green', 'yellow', 'orange', 'red'])
airtime_labels = np.digitize(average_airtime.column('AIR_TIME nanmean'), airtime_bins) - 1
average_airtime = average_airtime.with_column('colors', airtime_colors[np.clip(airtime_labels, 0, len(airtime_colors)-1)])

# Creating Map

In [18]:
map_info = average_airtime.select('LATITUDE', 'LONGITUDE', 'AIRPORT', 'count', 'colors')
map_info = map_info.relabel('AIRPORT', 'labels')
map_info = map_info.with_column('areas', map_info.column('count') * 0.03)

In [19]:
Circle.map_table(map_info.select('LATITUDE', 'LONGITUDE', 'labels', 'colors', 'areas'))

# Problem 2

Explain interesting aspects of what your map(s) reveals. In other words, what have you learned
from the map(s)?

- The map highlights Chicago O’Hare, Dallas/Ft. Worth, and Miami as the busiest American Airlines and American Eagle hubs, with Miami’s high flight volume likely influenced by international routes. Airports in remote locations like Anchorage show longer average airtimes, while Hawaiian airports have relatively smaller circles, suggesting shorter inter-island flights. The Eastern U.S. has denser flight activity, whereas the Western and Midwest regions have fewer but longer flights, indicating a hub-and-spoke network.